# 02 — Audio training

Slim copy from Drive (no `.venv`). Classical RF/XGB runs on CPU; LSTM needs GPU.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/content/hf_cache"
os.chdir("/content")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/audio-sentiment-analysis")
WORK = Path("/content/audio-sentiment-analysis")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()

for name in ["requirements.txt", "README.md", "TRAINING.md"]:
    if (DRIVE / name).exists():
        shutil.copy2(DRIVE / name, WORK / name)

for name in ["src", "configs"]:
    shutil.copytree(
        DRIVE / name, WORK / name,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".ipynb_checkpoints"),
    )

# Copy wavs only (skip huge non-audio junk)
for ds in ["crema_d", "banglaser"]:
    src = DRIVE / "data" / "audio" / ds
    dst = WORK / "data" / "audio" / ds
    if not src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        continue
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for wav in src.rglob("*.wav"):
        rel = wav.relative_to(src)
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(wav, target)
        n += 1
    print(f"copied {n} wavs -> data/audio/{ds}")

os.chdir(WORK)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import torch
HAS_GPU = torch.cuda.is_available()
print("device:", "cuda" if HAS_GPU else "cpu")
print("wav crema", len(list(Path("data/audio/crema_d").rglob("*.wav"))))
print("wav bangla", len(list(Path("data/audio/banglaser").rglob("*.wav"))))
if not HAS_GPU:
    print("No GPU: you can still run classical RF/XGB. Enable T4 before train_dnn.")

In [ ]:
import os
os.chdir("/content/audio-sentiment-analysis")
assert len(list(__import__('pathlib').Path('data/audio/crema_d').rglob('*.wav'))) > 0, "Put CREMA-D wavs in Drive data/audio/crema_d first"
!python -m src.audio.train_classical --dataset crema_d

In [ ]:
import os, torch
os.chdir("/content/audio-sentiment-analysis")
assert torch.cuda.is_available(), "Enable T4 GPU for LSTM"
!python -m src.audio.train_dnn --dataset crema_d --epochs 20 --batch-size 32

In [ ]:
import os
from pathlib import Path
os.chdir("/content/audio-sentiment-analysis")
if len(list(Path('data/audio/banglaser').rglob('*.wav'))) > 0:
    !python -m src.audio.train_classical --dataset banglaser
    import torch
    if torch.cuda.is_available():
        !python -m src.audio.train_dnn --dataset banglaser --epochs 20 --batch-size 16
else:
    print('Skip BanglaSER — no wavs')

In [ ]:
import os, shutil
from pathlib import Path
os.chdir("/content/audio-sentiment-analysis")
DRIVE_FINAL = Path("/content/drive/MyDrive/audio-sentiment-analysis/models_final/audio")
if DRIVE_FINAL.exists():
    shutil.rmtree(DRIVE_FINAL)
copied = 0
for p in Path('models/audio').rglob('*'):
    if not p.is_file() or 'runs' in p.parts:
        continue
    dest = DRIVE_FINAL / p.relative_to('models/audio')
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest)
    copied += 1
print('copied', copied, '->', DRIVE_FINAL)
!du -sh "{DRIVE_FINAL}"